# Data Preprocessing — Cleaning Stage
**Project 2 · Person 3** — Rossmann Store Sales

Goal: take the raw files (`train.csv`, `test.csv`, `store.csv`) and produce a clean, consistent dataset.

Steps:
1. Missing values
2. Duplicates
3. Inconsistencies (categories, invalid values)
4. Date column → datetime
5. Data types
6. Final validation
7. Save cleaned files

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('../dataset')
OUT = DATA / 'cleaned'
OUT.mkdir(exist_ok=True)

train = pd.read_csv(DATA / 'train.csv', low_memory=False)
test  = pd.read_csv(DATA / 'test.csv')
store = pd.read_csv(DATA / 'store.csv')

raw_shapes = {'train': train.shape, 'test': test.shape, 'store': store.shape}
log = []   # cleaning log: (step, action, rows/cells affected)
raw_shapes

{'train': (1017209, 9), 'test': (41088, 8), 'store': (1115, 10)}

## 0. Initial inspection

In [2]:
for name, df in [('train', train), ('test', test), ('store', store)]:
    print(f'===== {name} {df.shape} =====')
    print(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique()}))
    print()

===== train (1017209, 9) =====
               dtype  missing  unique
Store          int64        0    1115
DayOfWeek      int64        0       7
Date             str        0     942
Sales          int64        0   21734
Customers      int64        0    4086
Open           int64        0       2
Promo          int64        0       2
StateHoliday     str        0       4
SchoolHoliday  int64        0       2

===== test (41088, 8) =====
                 dtype  missing  unique
Id               int64        0   41088
Store            int64        0     856
DayOfWeek        int64        0       7
Date               str        0      48
Open           float64       11       2
Promo            int64        0       2
StateHoliday       str        0       2
SchoolHoliday    int64        0       2

===== store (1115, 10) =====
                             dtype  missing  unique
Store                        int64        0    1115
StoreType                      str        0       4
Assortment    

## 1. Missing values

| Table | Column | Missing | Decision |
|---|---|---|---|
| test | `Open` | 11 | All belong to store 622. Fill with the store's normal weekly pattern: closed on Sunday, open otherwise. |
| store | `CompetitionDistance` | 3 | Fill with the **median** (robust: distribution is heavily right-skewed). |
| store | `CompetitionOpenSinceMonth/Year` | 354 | Date is simply not recorded. Fill with `0` = "unknown" (a made-up date would be misleading). |
| store | `Promo2SinceWeek/Year`, `PromoInterval` | 544 | Not really missing: these stores have `Promo2 = 0` (never joined), so the fields do not apply. Fill with `0` / `'NoPromo'`. |
| store | `CompetitionOpenSinceYear = 1900` | 1 | Impossible value (store 815) → treated as missing → `0`. |

In [3]:
# --- test.Open ---
print(test[test['Open'].isna()][['Id', 'Store', 'Date', 'DayOfWeek']])
print('\nStore 622 in train - share of open days per weekday:')
print(train[train['Store'] == 622].groupby('DayOfWeek')['Open'].mean().round(2).to_dict())

          Id  Store        Date  DayOfWeek
479      480    622  2015-09-17          4
1335    1336    622  2015-09-16          3
2191    2192    622  2015-09-15          2
3047    3048    622  2015-09-14          1
4759    4760    622  2015-09-12          6
5615    5616    622  2015-09-11          5
6471    6472    622  2015-09-10          4
7327    7328    622  2015-09-09          3
8183    8184    622  2015-09-08          2
9039    9040    622  2015-09-07          1
10751  10752    622  2015-09-05          6

Store 622 in train - share of open days per weekday:
{1: 0.96, 2: 0.99, 3: 0.98, 4: 0.94, 5: 0.96, 6: 1.0, 7: 0.0}


In [4]:
n = test['Open'].isna().sum()
test['Open'] = test['Open'].fillna((test['DayOfWeek'] != 7).astype(int)).astype(int)
log.append(('Missing', 'test.Open filled (closed on Sunday, else open)', int(n)))

# --- store: impossible competition year ---
bad_year = store['CompetitionOpenSinceYear'] == 1900
print('Impossible competition years:', store.loc[bad_year, ['Store', 'CompetitionOpenSinceYear']].values.tolist())
store.loc[bad_year, ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']] = np.nan
log.append(('Invalid', 'CompetitionOpenSinceYear=1900 set to unknown', int(bad_year.sum())))

# --- store: CompetitionDistance ---
n = store['CompetitionDistance'].isna().sum()
med = store['CompetitionDistance'].median()
store['CompetitionDistance'] = store['CompetitionDistance'].fillna(med)
log.append(('Missing', f'CompetitionDistance filled with median ({med:.0f})', int(n)))

# --- store: competition open date ---
cols = ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']
n = store[cols[0]].isna().sum()
store[cols] = store[cols].fillna(0).astype(int)
log.append(('Missing', 'CompetitionOpenSince Month/Year filled with 0 (unknown)', int(n)))

# --- store: Promo2 fields (structural missing, Promo2 == 0) ---
print(pd.crosstab(store['Promo2'], store['PromoInterval'].isna(), rownames=['Promo2'], colnames=['PromoInterval missing']))
n = store['PromoInterval'].isna().sum()
store[['Promo2SinceWeek', 'Promo2SinceYear']] = store[['Promo2SinceWeek', 'Promo2SinceYear']].fillna(0).astype(int)
store['PromoInterval'] = store['PromoInterval'].fillna('NoPromo')
log.append(('Missing', 'Promo2SinceWeek/Year=0 and PromoInterval="NoPromo" for stores without Promo2', int(n)))

print('\nRemaining missing -> train:', train.isna().sum().sum(), '| test:', test.isna().sum().sum(), '| store:', store.isna().sum().sum())

Impossible competition years: [[815.0, 1900.0]]
PromoInterval missing  False  True 
Promo2                             
0                          0    544
1                        571      0

Remaining missing -> train: 0 | test: 0 | store: 0


## 2. Duplicates

In [5]:
for name, df, key in [('train', train, ['Store', 'Date']), ('test', test, ['Store', 'Date']), ('store', store, ['Store'])]:
    print(f'{name}: full-row duplicates = {df.duplicated().sum()}, duplicate keys {key} = {df.duplicated(key).sum()}')

for name, df in [('train', train), ('test', test), ('store', store)]:
    n = df.duplicated().sum()
    if n:
        df.drop_duplicates(inplace=True)
    log.append(('Duplicates', f'{name}: duplicate rows removed', int(n)))

train: full-row duplicates = 0, duplicate keys ['Store', 'Date'] = 0
test: full-row duplicates = 0, duplicate keys ['Store', 'Date'] = 0
store: full-row duplicates = 0, duplicate keys ['Store'] = 0


No duplicates were found. Each `(Store, Date)` pair is unique, so no records were dropped.

## 3. Inconsistencies

### 3.1 Category names
- `PromoInterval` uses `Sept` while the other months are 3-letter abbreviations → standardise to `Sep`.
- Strip whitespace and check letter case in all categorical columns.
- `StateHoliday` mixes numeric-looking `0` with letters (`a`/`b`/`c`) → keep everything as strings.

In [6]:
for df in (train, test):
    df['StateHoliday'] = df['StateHoliday'].astype(str).str.strip()

for col in ['StoreType', 'Assortment']:
    store[col] = store[col].astype(str).str.strip().str.lower()

before = store['PromoInterval'].value_counts().to_dict()
store['PromoInterval'] = store['PromoInterval'].astype(str).str.strip().str.replace('Sept', 'Sep', regex=False)
log.append(('Inconsistency', "PromoInterval 'Sept' -> 'Sep'", int(store['PromoInterval'].str.contains('Sep').sum())))

print('PromoInterval before:', before)
print('PromoInterval after :', store['PromoInterval'].value_counts().to_dict())
print('StateHoliday train  :', sorted(train['StateHoliday'].unique()), '| test:', sorted(test['StateHoliday'].unique()))
print('StoreType           :', sorted(store['StoreType'].unique()), '| Assortment:', sorted(store['Assortment'].unique()))

PromoInterval before: {'NoPromo': 544, 'Jan,Apr,Jul,Oct': 335, 'Feb,May,Aug,Nov': 130, 'Mar,Jun,Sept,Dec': 106}
PromoInterval after : {'NoPromo': 544, 'Jan,Apr,Jul,Oct': 335, 'Feb,May,Aug,Nov': 130, 'Mar,Jun,Sep,Dec': 106}
StateHoliday train  : ['0', 'a', 'b', 'c'] | test: ['0', 'a']
StoreType           : ['a', 'b', 'c', 'd'] | Assortment: ['a', 'b', 'c']


### 3.2 Invalid values
Checks: binary columns contain only 0/1, no negative `Sales`/`Customers`, `DayOfWeek` in 1–7, competition distance positive, every store in train/test exists in `store.csv`, and logical consistency between `Open`, `Sales` and `Customers`.

In [7]:
for col in ['Open', 'Promo', 'SchoolHoliday']:
    for name, df in [('train', train), ('test', test)]:
        assert set(df[col].unique()) <= {0, 1}, (name, col)
assert train['DayOfWeek'].between(1, 7).all() and test['DayOfWeek'].between(1, 7).all()
assert (train[['Sales', 'Customers']] >= 0).all().all()
assert (store['CompetitionDistance'] > 0).all()
assert set(train['Store']) <= set(store['Store']) and set(test['Store']) <= set(store['Store'])
print('Basic range/domain checks passed.\n')

print('Closed but Sales > 0        :', ((train['Open'] == 0) & (train['Sales'] > 0)).sum())
print('Open but Sales == 0         :', ((train['Open'] == 1) & (train['Sales'] == 0)).sum())
print('Open but Customers == 0     :', ((train['Open'] == 1) & (train['Customers'] == 0)).sum())
print('Sales > 0 but Customers == 0:', ((train['Sales'] > 0) & (train['Customers'] == 0)).sum())

Basic range/domain checks passed.

Closed but Sales > 0        : 0
Open but Sales == 0         : 54
Open but Customers == 0     : 52
Sales > 0 but Customers == 0: 0


54 rows say the store was **open** yet recorded **zero sales** (52 of them also have zero customers), across 41 different stores. This is contradictory (most likely a temporary closure/refurbishment logged as open), and it would distort the sales distribution. Since it is only ~0.005% of the data, we **remove these rows**.

Rows with `Open = 0` and `Sales = 0` are valid (the store was closed) and are **kept** — the modelling stage can decide to filter them.

In [8]:
bad = (train['Open'] == 1) & (train['Sales'] == 0)
train = train.loc[~bad].copy()
log.append(('Invalid', 'train: rows with Open=1 but Sales=0 removed', int(bad.sum())))
print('Removed', bad.sum(), 'rows -> train shape', train.shape)

Removed 54 rows -> train shape (1017155, 9)


## 4. Date column
Convert `Date` to `datetime64`, using an explicit format so anything that does not parse becomes an error, and verify that `DayOfWeek` agrees with the date.

In [9]:
for name, df in [('train', train), ('test', test)]:
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')
    print(f'{name}: failed conversions = {df["Date"].isna().sum()}, range = {df["Date"].min().date()} -> {df["Date"].max().date()}')
    mismatch = ((df['Date'].dt.dayofweek + 1) != df['DayOfWeek']).sum()
    print(f'{name}: DayOfWeek mismatches = {mismatch}')
    assert df['Date'].isna().sum() == 0 and mismatch == 0

train = train.sort_values(['Date', 'Store']).reset_index(drop=True)
test  = test.sort_values('Id').reset_index(drop=True)

train: failed conversions = 0, range = 2013-01-01 -> 2015-07-31
train: DayOfWeek mismatches = 0
test: failed conversions = 0, range = 2015-08-01 -> 2015-09-17
test: DayOfWeek mismatches = 0


## 5. Data types
- Identifiers and counts → integers
- `Open` in test was `float` (because of the NaNs) → integer
- Categorical text columns → `category`
- `StateHoliday` uses a fixed category set so train and test are consistent

In [10]:
from pandas.api.types import CategoricalDtype
holiday_type = CategoricalDtype(['0', 'a', 'b', 'c'])

for df in (train, test):
    df['StateHoliday'] = df['StateHoliday'].astype(holiday_type)

for col in ['StoreType', 'Assortment', 'PromoInterval']:
    store[col] = store[col].astype('category')

print(train.dtypes, '\n'); print(test.dtypes, '\n'); print(store.dtypes)

Store                     int64
DayOfWeek                 int64
Date             datetime64[us]
Sales                     int64
Customers                 int64
Open                      int64
Promo                     int64
StateHoliday           category
SchoolHoliday             int64
dtype: object 

Id                        int64
Store                     int64
DayOfWeek                 int64
Date             datetime64[us]
Open                      int64
Promo                     int64
StateHoliday           category
SchoolHoliday             int64
dtype: object 

Store                           int64
StoreType                    category
Assortment                   category
CompetitionDistance           float64
CompetitionOpenSinceMonth       int64
CompetitionOpenSinceYear        int64
Promo2                          int64
Promo2SinceWeek                 int64
Promo2SinceYear                 int64
PromoInterval                category
dtype: object


## 6. Final validation

In [11]:
tables = {'train': train, 'test': test, 'store': store}
report = pd.DataFrame({
    'raw_rows':      [raw_shapes[k][0] for k in tables],
    'clean_rows':    [len(v) for v in tables.values()],
    'missing_cells': [int(v.isna().sum().sum()) for v in tables.values()],
    'duplicate_rows':[int(v.duplicated().sum()) for v in tables.values()],
}, index=tables.keys())
display(report)

assert report['missing_cells'].sum() == 0
assert report['duplicate_rows'].sum() == 0
assert not train.duplicated(['Store', 'Date']).any() and not test.duplicated(['Store', 'Date']).any()
assert pd.api.types.is_datetime64_any_dtype(train['Date']) and pd.api.types.is_datetime64_any_dtype(test['Date'])
assert pd.api.types.is_integer_dtype(test['Open'])
print('All validation checks passed.')

,raw_rows,clean_rows,missing_cells,duplicate_rows
train,1017209,1017155,0,0
test,41088,41088,0,0
store,1115,1115,0,0


All validation checks passed.


In [12]:
print(pd.DataFrame(log, columns=['Type', 'Action', 'Count']).to_string(index=False))

         Type                                                                       Action  Count
      Missing                               test.Open filled (closed on Sunday, else open)     11
      Invalid                                 CompetitionOpenSinceYear=1900 set to unknown      1
      Missing                                CompetitionDistance filled with median (2325)      3
      Missing                      CompetitionOpenSince Month/Year filled with 0 (unknown)    355
      Missing Promo2SinceWeek/Year=0 and PromoInterval="NoPromo" for stores without Promo2    544
   Duplicates                                                train: duplicate rows removed      0
   Duplicates                                                 test: duplicate rows removed      0
   Duplicates                                                store: duplicate rows removed      0
Inconsistency                                                PromoInterval 'Sept' -> 'Sep'    106
      Invalid       

## 7. Save cleaned dataset

In [13]:
train.to_csv(OUT / 'cleaned_train.csv', index=False)
test.to_csv(OUT / 'cleaned_test.csv', index=False)
store.to_csv(OUT / 'cleaned_store.csv', index=False)

# sanity check: reload from disk
chk = pd.read_csv(OUT / 'cleaned_train.csv', parse_dates=['Date'])
assert chk.shape == train.shape and chk.isna().sum().sum() == 0
chk_store = pd.read_csv(OUT / 'cleaned_store.csv')
chk_test  = pd.read_csv(OUT / 'cleaned_test.csv', parse_dates=['Date'])
assert chk_store.isna().sum().sum() == 0 and chk_test.isna().sum().sum() == 0
print('Saved:', [p.name for p in OUT.glob('cleaned_*.csv')])

Saved: ['cleaned_train.csv', 'cleaned_store.csv', 'cleaned_test.csv']


/tmp/ipykernel_177/3742839541.py:6: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  chk = pd.read_csv(OUT / 'cleaned_train.csv', parse_dates=['Date'])


## Summary of cleaning decisions
- **Missing values:** `test.Open` (11, store 622) filled by weekday pattern; `CompetitionDistance` (3) median; competition-open date (354) and Promo2 fields (544) filled with explicit "unknown / not applicable" markers.
- **Duplicates:** none found in any table.
- **Inconsistencies:** `Sept` → `Sep`; category text normalised; impossible year 1900 → unknown; 54 contradictory rows (`Open=1`, `Sales=0`) removed.
- **Date:** converted to datetime in train and test, verified against `DayOfWeek`.
- **Types:** integers, categories and datetimes set appropriately.

**Notes for the next stage**
- Closed days (`Open = 0`, `Sales = 0`) are kept; they are valid records.
- `0` in `CompetitionOpenSinceMonth/Year` and `Promo2Since*` means "unknown / not applicable" — handle it when engineering features.
- `cleaned_train.csv` and `cleaned_store.csv` can be joined on `Store`.
- When re-loading, use `pd.read_csv(..., parse_dates=['Date'], dtype={'StateHoliday': str})`. `PromoInterval` uses `'NoPromo'` (not `'None'`) because pandas reads the text `None` as NaN.